# Selective Checkpoint Transfer

Bizim tezimiz bir transfer öğrenme tezidir. Yani merkez sorumuz şu olacak "Spor yayınlarında eğitilmiş bir Türkçe video özetleme modeli, haber - diziye transfer olur mu ?

Fakat burada 2 farklı transfer vardır:
 - Domain Transfer: spor --> haber / dizi. bu kısım ileride model eğitildikten sonra olacak

 - Checkpoint Transfer: Ingilizce veri ile hazırlanmış bir modelden kendi modelimize bazı özelliklerin transferi. mrhisum.pth

Yani kendi oluşturacağımız modeli belirli bir aşamaya kadar gelmiş bir modelin temellerine oturtacağız.

## Peki neden sıfırdan başlamıyoruz ?

Elimizde sıfırdan bir model eğitecek kadar verimiz yok ve mrhisum.pth hali hazırda bazı şeyleri öğrenmiş bir model. Örn:
,
  - temporal blocklar
  - cross- modal fuzyon
  - windowed self attention

Bunlar dile ve içeriğin türüne göre oldukça bağımsızdır. Bu yüzden bu öğrenilmiş özellikleri direkt kendi modelimize transfer edebiliriz.

## Peki neden selective ?

Burada özellik kopyalama yaparken herşeyi direkt olduğu gibi kopyalamıyoruz. 2 özelliğimiz farklı olacak:

 1. **Text Encoder:** Orginal model text'i RoBERT'a ile encode ediyordu. Fakat bizim altyazılarımız Türkçe olacağı için BERTurk kullanacağız. ikisi de 768 boyutlu bir çıktı verir. O yüzden text_proj'ü kopyalayacağız, sonra fine-tune 'da BERTurk kullanacağız.

 2. **Görsel giriş boyutu:** Orjinal modelde visual feature 1024-d idi. Fakat InceptionV3 2048-d çıktı verir. Bu uyuşmaz. O yüzden visual_proj'ü kopyalamayacağız. Sıfırdan, bizim spor verimizden eğitilecek.

Geri kalan herşey direkt olarak öğrenilmiş bir modelden birebir alacağız.

Bu oturumun sonunda elimizde eğitime hazır bir model olacaktır.



## State Dict Nedir ?

**State Dict:** Bir PyTorch modelinin tüm öğrenilmiş veya öğrenilebilir ağırlıkları
  - katman_ismi --> tensör

eşleşmesi olarak tutulur. Tıpkı bir python sözlüğü gibi:

{'visual_proj.weight': tensor[128,1024], 'visual_proj.bias': tensor[128], 'temporal_block.0...': ...}

İşte checkpoint dosyası (.pth) de bu sözlüğün diske kaydedilmiş halidir. Yani biz bu sözlüğü okuyup kendi modelimizin katmanlarımıza dağıtacağız.

Bunun için **load_state_dict** methodunu kullanacağız. Bu method, katmanları isim isim eşleştirir. Checkpoint'teki 'visual_proj.weight' anahtarını, modelde direkt olarak 'visual_proj.weight' arar.
Eğer iki anahtar eşleşirse ekler. İsim modelde var cp'de yoksa missing, cp'de var model de yoksa unexpected olarak döner.

Bu modelde bir de **strict** parametresi var. Bunu True olarak ayarlarsak en ufak bir uyumsuzlukta hata döndürecektir. Biz bunu False olarak tutacağız. Çünkü beklentimiz uyanlar yüklenecek, uymayanlar atlanacak ve missing, unexpected listesine düşecek. Eğer boyut farkları çıkarsa False dahi olsa hata fırlatır.


In [ ]:
# Model kurulumu

import torch
from models.model import TripleSumm # bizim yazdığımız model

model = Triplesumm() # visiual_dim default 1024, bizim model 2048 olacak
print(sum(p.numel() for p in model.parameters()))

print("shape: ", visual_proj.weight.shape)

# çıktı:
# 1406337
# visual_proj shape: torch.Size([128, 1024])

In [ ]:
# checkpoint yükleme

path = .....
ckpt = torch.load("checkpoint/best_model_ckpt_mrhisum.pth", map_location="cpu",
                  weights_only=False)
# map_location = cpu -> cp GPU'da kayıtlı olabilir, CPU'ya zorluyoruz.

print(type(ckpt))
print(ckpt.keys())
# (visual_proj.weight, temporal_block.0..., head.4.bias ...)

result = model.load_state_dict(ckpt, strict=False)
print(result)
# çıktı: <All keys matched successfully>
# missing: [], unexpected: []  → 108 katmanın hepsi birebir oturdu


In [ ]:
# Kendi modelimiz

model_v2 = Triplesumm(visual_dim = 2048)
print(model_v2.visual_proj.weight.shape)
# torch.Size([128, 2048]) ->  bizim modelimizin dimension 2048.
# yukarıda bahsettik bu uyumsuzluğu

result_v2 = model_v2.load_state_dict(ckpt, strict=False)
print(result_v2)
# RuntimeError: Error(s) in loading state_dict for TripleSumm:
# size mismatch for visual_proj.weight: copying a param with shape
# torch.Size([128, 1024]) from checkpoint, the shape in current model
# is torch.Size([128, 2048]).

# yani strict=False boyut uyuşmazlığından dolayı hata verdi.
# visiual_proj 1024 beklenirken bizim modelimiz 2048 döndürdü.
# bu yüzden load_state_dict'ten önce cp' kopyasından visiual_proj'u cıkartacağız.


In [ ]:
# uyusmayan kısmı filtreleme
ckpt_filtered = dict(ckpt)

# uyusmayanları cıkarıyoruz
del ckpt_filtered["visual_proj.weight"]
del ckpt_filtered["visual_proj.bias"]

result_v2 = model_v2.load_state_dict(ckpt_filtered, strict=False)
# _IncompatibleKeys(
#   missing_keys=['visual_proj.weight', 'visual_proj.bias'],
#   unexpected_keys=[]

# missing: sadece visual_proj → bizim 2048'lik yeni katmanımız,
# rastgele başlangıçta, fine-tune'da öğrenecek
# unexpected: boş → checkpoint'te modele giremeyen fazlalık yok
# görünmeyen ama en önemli: diğer 106 katman sessizce, birebir transfer oldu


In [ ]:
# Test

print(torch.allclose(model_v2.head[4].weight, ckpt["head.4.weight"]))
# True  → head.4 birebir checkpoint'ten geldi, transfer gerçek

print(model_v2.visual_proj.weight.shape)   # torch.Size([128, 2048])
print(model_v2.visual_proj.weight.mean())  # ~3.3e-05
print(model_v2.visual_proj.weight.std())   # ~0.0128

# Şekil [128,2048] → checkpoint'in 1024'ünden farklı, yani checkpoint'ten gelmedi.

# mean~0, std~0.013 → sağlıklı rastgele başlangıç (ne ölü/sıfır, ne kaymış).
# nn.Linear'ın default init'i böyle.

# grad_fn'in var olması → katman öğrenilebilir durumda (donmuş değil).

In [ ]:
# dummy veri ile bir forward deneyelim.

visual = torch.randn(1, 100, 2048)
text   = torch.randn(1, 100, 768)
audio  = torch.randn(1, 100, 768)

model_v2.eval()# dropout'u kapatır.

with torch.no_grad(): # gradyan hesabını atlar, sadece test.
    out, attn_weights = model_v2(visual, text, audio)

print(out.shape)              # torch.Size([1, 100])
print(out.min(), out.max())   # tensor(0.2303) tensor(0.5193)

